## Install and import dependencies

First we install the OpenAI python package, which let's us make easy requests to OpenAI API compatible endpoints, which models served in OpenShift AI using vLLM are by default.

In [ ]:
!pip install openai

In [ ]:
import mlflow
from openai import OpenAI
import os

import warnings
warnings.filterwarnings("ignore")

## Configuration

After we have our dependencies we need to configure our environment to use MLflow. Here we set up the experiment, which prompt to use, and that we want to automatically log all requests that uses the openai sdk we installed before.  
In the second cell we set up our LLM client we will use for sending requests to the model.

In [ ]:
# ─────────────────────────────────────────────────────────────────
# CONFIGURATION — update these to match your environment
# ─────────────────────────────────────────────────────────────────

# MLflow server
MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI")
MLFLOW_TRACKING_TOKEN = ""

# MLflow names (change if you want to)
EXPERIMENT_NAME = "hospital-helpdesk"
PROMPT_NAME = "ai-hospital-helpdesk"

# Connect to MLflow
os.environ["MLFLOW_TRACKING_TOKEN"] = MLFLOW_TRACKING_TOKEN
os.environ["MLFLOW_WORKSPACE"] = "hospital-helpdesk"
os.environ["MLFLOW_TRACKING_INSECURE_TLS"] = "true"

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

if experiment is None:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)
else:
    experiment_id = experiment.experiment_id

experiment_id = mlflow.set_experiment(EXPERIMENT_NAME)

mlflow.openai.autolog()

In [ ]:
# LLM for inference
LLM_API_KEY = "dummy_key"
LLM_BASE_URL = "http://tinyllamatinyllama-11b-chat-v1-predictor.hospital-helpdesk.svc.cluster.local:8080"
LLM_MODEL = "tinyllamatinyllama-11b-chat-v1"

client = OpenAI(
    base_url=LLM_BASE_URL + "/v1",
    api_key=LLM_API_KEY
)

In [ ]:
sys_prompt_mlflow = mlflow.genai.load_prompt(f"prompts:/{PROMPT_NAME}@latest")
sys_prompt_plaintext = next(m["content"] for m in sys_prompt_mlflow.format() if m["role"] == "system")
sys_prompt_plaintext

## Send the requests

We then create a small send_request function that simply sends a message, using our system prompt from before, and streams back the answer.  
We have a list of message that we send to the model using this function, since the model is running on CPU, it may take a few minutes to get responses for all of these requests.

In [ ]:
def send_request(message):
    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": sys_prompt_plaintext},
            {"role": "user", "content": message}
        ],
        stream=True,
    )
    
    for chunk in response:
        if hasattr(chunk, 'choices') and len(chunk.choices) > 0:
            delta = chunk.choices[0].delta
            if hasattr(delta, 'content') and delta.content:
                print(delta.content, end="", flush=True)
    print()

In [ ]:
messages = [
    "A patient just called asking how to get to the radiology department from the main entrance. Can you help me draft a response?",
    "What are the typical visiting hours policies for ICU patients in most hospitals?",
    "Can you give me a quick summary of what the oncology department typically handles, so I can explain it to a caller?",
    "A caller is getting frustrated because they've been transferred three times. How should I handle this situation?",
    "What information should I collect from a caller before transferring them to the surgical ward?"
]

In [ ]:
for message in messages:
    send_request(message)
    print("------------------------------------------------------------------------------------------------------------")